In [ ]:
year = 2024

In [ ]:
import csv

def is_overall_data(s):
    """Check if a string is a score, penalty, or 'pres only' status."""
    if s.lower() in ['pres', 'only']:
        return True
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_overall_results(input_filepath, output_filepath):
    # There are 10 possible trailing data columns:
    # Penalty, Cost, Presentation, Design, Accel, Skidpad, Autocross, Endurance, Efficiency, Total
    headers = [
        "Place", "Car Num", "Team", 
        "Penalty", "Cost Score", "Presentation Score", "Design Score", "Acceleration Score", 
        "Skid Pad Score", "Autocross Score", "Endurance Score", "Efficiency Score", "Total Score"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines, headers, or table artifacts
            if not line or line.startswith("Place") or line.startswith("Score") or line.startswith("Design") or not line[0].isdigit():
                continue
            
            parts = line.split()
            if not parts:
                continue

            # 1. Parse Place and Car Num based on formatting variations
            if len(parts) >= 3 and parts[0].isdigit() and parts[1] == 'T' and parts[2].isdigit():
                # Handles ties (e.g., "106 T 114")
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            elif len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
                # Handles standard ranked teams (e.g., "1 65")
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]
            elif len(parts) >= 1 and parts[0].isdigit():
                # Handles unranked teams with no place (e.g., "25 Instituto...")
                place = "Unranked"
                car_num = parts[0]
                parts = parts[1:]
            else:
                continue

            # 2. Vacuum up all scores, penalties, and statuses from the back
            tail_data = []
            while parts and is_overall_data(parts[-1]):
                tail_data.insert(0, parts.pop())

            # 3. Whatever is left over perfectly reconstructs the Team Name
            team_name = " ".join(parts)

            # 4. Pad the tail data so it lines up perfectly in the CSV columns
            tail_data += [''] * (10 - len(tail_data))

            # Write the formatted row to the CSV
            row = [place, car_num, team_name] + tail_data
            writer.writerow(row)

    print(f"Success! Overall Results data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_overall_results(f"{year}-raw-overall.txt", 'fsae_overall_results.csv')

Success! Overall Results data parsed and saved to fsae_overall_results.csv


In [2]:
import csv

def is_score(s):
    """Check if a string is a number."""
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_design_results(input_filepath, output_filepath):
    # Match the exact headers from your image
    headers = [
        "Place", "Car Num", "Team", "Document Penalty", 
        "Raw Score", "Late Penalty", "Status", "Score"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines or stray header fragments
            if not line or line.startswith("Place") or line.startswith("Score") or line.startswith("Document"):
                continue
            
            parts = line.split()
            
            # Ensure it's a valid data row (must start with a ranking number)
            if not parts or not parts[0][0].isdigit():
                continue
            
            # 1. Parse Place and Car Num from the front
            if len(parts) >= 3 and parts[0].isdigit() and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            elif len(parts) >= 2 and parts[0].isdigit():
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]
            else:
                continue

            # 2. Parse from the back to find Status and Score
            score = parts.pop()
            status = ""
            
            # If the item right before the score is NOT a number (e.g., "Final", "DNA", "RFP"), it's the Status
            if parts and not is_score(parts[-1]):
                status = parts.pop()
                
            # 3. Collect the remaining numbers (Penalties and Raw Score)
            tail_nums = []
            while parts and is_score(parts[-1]):
                tail_nums.insert(0, parts.pop())
                
            # Whatever is left over is the full team name
            team_name = " ".join(parts)
            
            # 4. Use math to route the numbers to the correct columns
            doc_pen = ""
            raw_score = ""
            late_pen = ""
            
            if len(tail_nums) == 1:
                # If there's only one number, it has to be the Raw Score
                raw_score = tail_nums[0]
                
            elif len(tail_nums) == 2:
                val1 = float(tail_nums[0])
                val2 = float(tail_nums[1])
                final_s = float(score)
                
                # Math Check A: Raw Score - Late Penalty = Score
                if abs(val1 - val2 - final_s) < 0.01:
                    raw_score = tail_nums[0]
                    late_pen = tail_nums[1]
                
                # Math Check B: Raw Score - Document Penalty = Score
                # (Formula is inverted here because Document Penalty would appear first in the text)
                elif abs(val2 - val1 - final_s) < 0.01:
                    doc_pen = tail_nums[0]
                    raw_score = tail_nums[1]
                else:
                    # Fallback if math is weird
                    raw_score = tail_nums[0]
                    late_pen = tail_nums[1]
                    
            elif len(tail_nums) >= 3:
                # If there are three numbers, all columns are filled
                doc_pen = tail_nums[0]
                raw_score = tail_nums[1]
                late_pen = tail_nums[2]

            # 5. Write the perfectly formatted row to the CSV
            row = [place, car_num, team_name, doc_pen, raw_score, late_pen, status, score]
            writer.writerow(row)

    print(f"Success! Data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_design_results('./raw-design-results.txt', 'fsae_design_results.csv')

Success! Data parsed and saved to fsae_design_results.csv


In [3]:
import csv

def is_score(s):
    """Check if a string is a number, gracefully handling stray asterisks."""
    s = s.replace('*', '').strip()
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_presentation_results(input_filepath, output_filepath):
    # We add "Status" to account for the unnamed column in the PDF
    headers = ["Place", "Car Num", "Team", "Status", "Raw Score", "Penalty", "Score"]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines and header fragments
            if not line or line.startswith("Place") or line.startswith("Raw") or line.startswith("Design") or line.startswith("Score") or line.startswith("Acceleration"):
                continue
            
            parts = line.split()
            
            # Ensure it's a valid data row (must start with a ranking number)
            if not parts or not parts[0][0].isdigit():
                continue
            
            # Clean up stray asterisks (like the one at the end of row 30)
            if parts[-1] == '*':
                parts.pop()
            parts[-1] = parts[-1].replace('*', '')

            # 1. Parse Place and Car Num
            if len(parts) >= 3 and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            else:
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]

            # 2. Extract scores from the back
            tail_nums = []
            while parts and is_score(parts[-1]):
                tail_nums.insert(0, parts.pop())

            # 3. Extract Status (if present)
            status = ""
            # FSAE often uses "Final", "DNA", "DNF", "RFP" in this column
            if parts and parts[-1].upper() in ["FINAL", "RFP", "DNA", "DNF"]:
                status = parts.pop()

            # 4. What's left is the Team Name
            team_name = " ".join(parts)

            # 5. Route the scores into the correct columns
            raw_score = ""
            penalty = ""
            score = ""

            if len(tail_nums) == 1:
                score = tail_nums[0]
            elif len(tail_nums) == 2:
                # If two numbers, it's almost always Raw Score and Score (Penalty is blank)
                raw_score = tail_nums[0]
                score = tail_nums[1]
            elif len(tail_nums) >= 3:
                # If three numbers, we have Raw Score, Penalty, and Score
                raw_score = tail_nums[0]
                penalty = tail_nums[1]
                score = tail_nums[2]

            # Write the beautifully formatted row
            row = [place, car_num, team_name, status, raw_score, penalty, score]
            writer.writerow(row)

    print(f"Success! Presentation data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_presentation_results('raw-presentation.txt', 'fsae_presentation_results.csv')

Success! Presentation data parsed and saved to fsae_presentation_results.csv


In [4]:
import csv

def parse_cost_results(input_filepath, output_filepath):
    # Match the exact headers from your image
    headers = [
        "Place", "Car Num", "Team", "Adjusted Cost", 
        "Price Score (30)", "Cost Accuracy (15)", "Engineering Drawings (15)", 
        "Scenario Score (40)", "Penalty", "Score"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines or header fragments
            if not line or line.startswith("Place") or line.startswith("1 119 Kennesaw") == False and not line[0].isdigit():
                # Just a safeguard to ensure we only process lines starting with a digit
                if not line[0].isdigit():
                    continue
            
            parts = line.split()
            if not parts:
                continue

            # Find the index of the Adjusted Cost (the string containing '$')
            cost_index = -1
            for i, part in enumerate(parts):
                if '$' in part:
                    cost_index = i
                    break
            
            # If for some reason a line doesn't have a '$', skip it to avoid crashing
            if cost_index == -1:
                continue

            # 1. Parse Place and Car Num from the front
            if parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                team_start_idx = 3
            else:
                place = parts[0]
                car_num = parts[1]
                team_start_idx = 2

            # 2. Extract Team Name (everything between Car Num and the '$')
            team_name = " ".join(parts[team_start_idx:cost_index])
            
            # 3. Extract Adjusted Cost
            adjusted_cost = parts[cost_index]

            # 4. Extract the trailing scores
            tail_nums = parts[cost_index + 1:]
            
            price_score = ""
            cost_acc = ""
            eng_draw = ""
            scenario = ""
            penalty = ""
            score = ""

            # Route the scores based on how many exist
            if len(tail_nums) == 5:
                # No penalty column
                price_score, cost_acc, eng_draw, scenario, score = tail_nums
            elif len(tail_nums) == 6:
                # Penalty is included
                price_score, cost_acc, eng_draw, scenario, penalty, score = tail_nums
            else:
                # Fallback just in case teams DNF'd and have missing middle scores
                if tail_nums:
                    score = tail_nums[-1]

            # Write the formatted row to the CSV
            row = [place, car_num, team_name, adjusted_cost, price_score, cost_acc, eng_draw, scenario, penalty, score]
            writer.writerow(row)

    print(f"Success! Cost Event data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_cost_results('raw-cost.txt', 'fsae_cost_results.csv')

Success! Cost Event data parsed and saved to fsae_cost_results.csv


In [8]:
import csv

def is_run_data(s):
    """Check if a string is a time, score, cone count, or a DNA/DNF status."""
    if s.upper() in ['DNA', 'DNF']:
        return True
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_acceleration_results(input_filepath, output_filepath):
    # Because runs collapse, we use generic data bins for the trailing numbers
    headers = [
        "Place", "Car Num", "Team", 
        "Run 1 Time", "Run 1 Adj Time", "Run 2 Time", "Run 2 Adj Time", "Run 3 Time", 
        "Run 3 Adj Time", "Run 4 Time", "Run 4 Adj Time", "Final Best Time", "Final Score"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines, headers, or lines that don't start with a ranking number
            if not line or line.startswith("Place") or line.startswith("Time") or not line[0].isdigit():
                continue
            
            parts = line.split()
            if not parts:
                continue

            # 1. Parse Place and Car Num from the front
            # (Includes a safeguard just in case there are ties formatted as "11 T")
            if len(parts) >= 3 and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            else:
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]

            # 2. Vacuum up all the times, cones, and DNA/DNF statuses from the back
            tail_data = []
            while parts and is_run_data(parts[-1]):
                tail_data.insert(0, parts.pop())

            # 3. Whatever is left over perfectly reconstructs the Team Name
            team_name = " ".join(parts)

            # 4. Pad the tail data so it lines up in the CSV columns
            tail_data += [''] * (11 - len(tail_data))

            # Write the formatted row to the CSV
            row = [place, car_num, team_name] + tail_data
            writer.writerow(row)

    print(f"Success! Acceleration Event data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_acceleration_results('raw-accel.txt', 'fsae_accel_results.csv')

Success! Acceleration Event data parsed and saved to fsae_accel_results.csv


In [11]:
import csv

def is_skidpad_data(s):
    if s.upper() in ['DNA', 'DNF']: return True
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_run(tokens):
    """Eats tokens left-to-right to build a single run [Time R, Time L, Cones, AdjTime]"""
    if not tokens:
        return ['', '', '', ''], []

    # 1. Check for immediate DNA/DNF (e.g., team skipped the run)
    if tokens[0].upper() in ['DNA', 'DNF']:
        return ['', '', '', tokens[0]], tokens[1:]

    # 2. Look ahead for an interrupting DNF (e.g., "5.347 DNF" or "5.803 5.420 DNF")
    for i in range(1, min(4, len(tokens))):
        if tokens[i].upper() in ['DNA', 'DNF']:
            run_data = [''] * 4
            run_data[3] = tokens[i]
            for j in range(i):
                run_data[j] = tokens[j]
            return run_data, tokens[i+1:]

    # 3. Math check for a clean run (3 numbers: R, L, Adj)
    if len(tokens) >= 3:
        try:
            r = float(tokens[0])
            l = float(tokens[1])
            adj3 = float(tokens[2])
            # Is (R + L) / 2 == AdjTime? (Using 0.05 for rounding leniency)
            if abs(((r + l) / 2) - adj3) <= 0.05: 
                return [tokens[0], tokens[1], '', tokens[2]], tokens[3:]
        except ValueError:
            pass

    # 4. Math check for a Cone Penalty run (4 numbers: R, L, Cones, Adj)
    if len(tokens) >= 4:
        try:
            r = float(tokens[0])
            l = float(tokens[1])
            cones = float(tokens[2])
            adj4 = float(tokens[3])
            # Is (R + L) / 2 + (Cones * 0.125) == AdjTime?
            if abs(((r + l) / 2 + (cones * 0.125)) - adj4) <= 0.05:
                return [tokens[0], tokens[1], tokens[2], tokens[3]], tokens[4:]
        except ValueError:
            pass

    # 5. Fallback: If math fails but we have numbers, assume it's a 3-token clean run.
    if len(tokens) >= 3:
         return [tokens[0], tokens[1], '', tokens[2]], tokens[3:]

    # Absolute fallback to prevent crashes
    return [tokens[0], '', '', ''], tokens[1:]


def parse_skidpad_results(input_filepath, output_filepath):
    headers = [
        "Place", "Car Num", "Team", 
        "D1 R1 Time R", "D1 R1 Time L", "D1 R1 Cones", "D1 R1 Adj Time",
        "D1 R2 Time R", "D1 R2 Time L", "D1 R2 Cones", "D1 R2 Adj Time",
        "D2 R1 Time R", "D2 R1 Time L", "D2 R1 Cones", "D2 R1 Adj Time",
        "D2 R2 Time R", "D2 R2 Time L", "D2 R2 Cones", "D2 R2 Adj Time",
        "Best Time", "Penalty", "Score"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            if not line or line.startswith("Place") or line.startswith("Time") or line.startswith("Driver"):
                continue
            
            parts = line.split()
            if not parts or not parts[0][0].isdigit():
                continue

            # 1. Parse Place and Car Num 
            if len(parts) >= 3 and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            else:
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]

            # 2. Vacuum up the tail data, but keep it in left-to-right order!
            tail_data = []
            while parts and is_skidpad_data(parts[-1]):
                tail_data.insert(0, parts.pop())

            # 3. What's left is the Team Name
            team_name = " ".join(parts)

            # 4. Use the Math-Parser to eat through the tail data run by run
            run1, tail_data = parse_run(tail_data)
            run2, tail_data = parse_run(tail_data)
            run3, tail_data = parse_run(tail_data)
            run4, tail_data = parse_run(tail_data)

            # 5. Route remaining numbers to Best Time, Penalty, and Score
            best_time = ""
            penalty = ""
            score = ""
            
            if len(tail_data) == 1:
                score = tail_data[0]
            elif len(tail_data) == 2:
                best_time = tail_data[0]
                score = tail_data[1]
            elif len(tail_data) >= 3:
                best_time = tail_data[0]
                penalty = tail_data[1]
                score = tail_data[2]

            # 6. Combine everything and write to CSV
            row = [place, car_num, team_name] + run1 + run2 + run3 + run4 + [best_time, penalty, score]
            writer.writerow(row)

    print(f"Success! Smart Skidpad Event data parsed and saved to {output_filepath}")

if __name__ == "__main__":
    parse_skidpad_results('raw-skidpad.txt', 'fsae_skidpad_results.csv')

Success! Smart Skidpad Event data parsed and saved to fsae_skidpad_results.csv


In [12]:
import csv

def is_autocross_data(s):
    """Check if a string is a metric or a DNA/DNF status."""
    if s.upper() in ['DNA', 'DNF']: return True
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_autocross_run(tokens):
    """Eats tokens left-to-right to build a single run [Time, Cones, Off Course, AdjTime]"""
    if not tokens:
        return ['', '', '', ''], []

    # 1. Check for immediate DNA/DNF (Team skipped or did not finish the run)
    if tokens[0].upper() in ['DNA', 'DNF']:
        return ['', '', '', tokens[0]], tokens[1:]

    # 2. Case A: 4 tokens [Time, Cones, Off Course, AdjTime]
    # Math check: Time + (Cones * 2) + (OffCourse * 20) == AdjTime
    if len(tokens) >= 4:
        try:
            t = float(tokens[0])
            c = float(tokens[1])
            oc = float(tokens[2])
            adj = float(tokens[3])
            if abs((t + c*2 + oc*20) - adj) <= 0.05:
                return [tokens[0], tokens[1], tokens[2], tokens[3]], tokens[4:]
        except ValueError:
            pass

    # 3. Case B: 3 tokens [Time, Penalty, AdjTime]
    if len(tokens) >= 3:
        try:
            t = float(tokens[0])
            p = float(tokens[1])
            adj = float(tokens[2])
            
            # Sub-case B1: Is it a Cone penalty? (Time + Penalty*2 == AdjTime)
            if abs((t + p*2) - adj) <= 0.05:
                return [tokens[0], tokens[1], '', tokens[2]], tokens[3:]
            
            # Sub-case B2: Is it an Off Course penalty? (Time + Penalty*20 == AdjTime)
            elif abs((t + p*20) - adj) <= 0.05:
                return [tokens[0], '', tokens[1], tokens[2]], tokens[3:]
        except ValueError:
            pass

    # 4. Case C: 2 tokens [Time, AdjTime]
    # Math check: Time == AdjTime (A perfectly clean run)
    if len(tokens) >= 2:
        try:
            t = float(tokens[0])
            adj = float(tokens[1])
            if abs(t - adj) <= 0.05:
                return [tokens[0], '', '', tokens[1]], tokens[2:]
        except ValueError:
            pass

    # 5. Fallback: Prevent the script from crashing if the copy-paste got scrambled
    return [tokens[0], '', '', ''], tokens[1:]

def parse_autocross_smart(input_filepath, output_filepath):
    headers = [
        "Place", "Car Num", "Team", 
        "R1 Time", "R1 Cones", "R1 Off Course", "R1 Adj Time",
        "R2 Time", "R2 Cones", "R2 Off Course", "R2 Adj Time",
        "R3 Time", "R3 Cones", "R3 Off Course", "R3 Adj Time",
        "R4 Time", "R4 Cones", "R4 Off Course", "R4 Adj Time",
        "Best Time", "Penalty", "Score"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines, headers, or lines that don't start with a ranking
            if not line or line.startswith("Place") or line.startswith("Time") or not line[0].isdigit():
                continue
            
            parts = line.split()
            if not parts or not parts[0][0].isdigit():
                continue

            # 1. Parse Place and Car Num from the front 
            if len(parts) >= 3 and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            else:
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]

            # 2. Vacuum up the tail data, keeping it in strict left-to-right order!
            tail_data = []
            while parts and is_autocross_data(parts[-1]):
                tail_data.insert(0, parts.pop())

            # 3. What's left is the Team Name
            team_name = " ".join(parts)

            # 4. Use the Math-Parser to "eat" through the tail data run by run
            run1, tail_data = parse_autocross_run(tail_data)
            run2, tail_data = parse_autocross_run(tail_data)
            run3, tail_data = parse_autocross_run(tail_data)
            run4, tail_data = parse_autocross_run(tail_data)

            # 5. Route any remaining numbers to Best Time, Penalty, and Score
            best_time = ""
            penalty = ""
            score = ""
            
            if len(tail_data) == 1:
                score = tail_data[0]
            elif len(tail_data) == 2:
                best_time = tail_data[0]
                score = tail_data[1]
            elif len(tail_data) >= 3:
                best_time = tail_data[0]
                penalty = tail_data[1]
                score = tail_data[2]

            # 6. Combine everything into a beautifully formatted row
            row = [place, car_num, team_name] + run1 + run2 + run3 + run4 + [best_time, penalty, score]
            writer.writerow(row)

    print(f"Success! Smart Autocross Event data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_autocross_smart('raw-autocross.txt', 'fsae_autocross_results.csv')

Success! Smart Autocross Event data parsed and saved to fsae_autocross_results.csv


In [4]:
import csv

def is_endurance_data(s):
    """Check if a string is a time, score, lap count, penalty, or DNF/DNA status."""
    if s.upper() in ['DNA', 'DNF']:
        return True
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_endurance_results(input_filepath, output_filepath):
    # Using generic data bins because up to 3 penalty columns can collapse, 
    # and DNFs drop several trailing score columns. 
    # Maximum possible trailing data points is 9.
    headers = [
        "Place", "Car Num", "Team", 
        "Data 1", "Data 2", "Data 3", "Data 4", "Data 5", 
        "Data 6", "Data 7", "Data 8", "Data 9"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines, headers, or lines that don't start with a ranking number
            if not line or line.startswith("Place") or line.startswith("Time") or not line[0].isdigit():
                continue
            
            parts = line.split()
            if not parts:
                continue

            # 1. Parse Place and Car Num from the front (handling "T" for ties)
            if len(parts) >= 3 and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            else:
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]

            # 2. Vacuum up all times, laps, penalties, scores, and DNF statuses from the back
            tail_data = []
            while parts and is_endurance_data(parts[-1]):
                tail_data.insert(0, parts.pop())

            # 3. Whatever is left over perfectly reconstructs the Team Name
            team_name = " ".join(parts)

            # 4. Pad the tail data so it lines up in the CSV columns
            tail_data += [''] * (9 - len(tail_data))

            # Write the formatted row to the CSV
            row = [place, car_num, team_name] + tail_data
            writer.writerow(row)

    print(f"Success! Endurance Event data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_endurance_results('raw-endurance.txt', 'fsae_endurance_results.csv')

Success! Endurance Event data parsed and saved to fsae_endurance_results.csv


In [ ]:
import csv

def is_efficiency_data(s):
    """Check if a string is a metric, DNF/DNA status, or the E85 fuel type."""
    # Explicitly catch text-based fuel types and statuses
    if s.upper() in ['DNA', 'DNF', 'E85']:
        return True
    # Catch all other numbers (including lap times, scores, and '93' or '100' octane)
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_efficiency_results(input_filepath, output_filepath):
    # We use 8 generic data bins to catch the trailing metrics 
    # (Laptime, Laps, Fuel Used, CO2, CO2/Lap, Fuel Type, Factor, Score)
    headers = [
        "Place", "Car Num", "Team", 
        "Average Adjusted Laptime", "Laps Completed", "Fuel Used (ltr)", "Adjusted CO2 (kg)", 
        "Average Adjusted CO2 per Lap", "Fuel Type", "Fuel Efficiency Factor", "Fuel Efficiency Score"
    ]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines, headers, or table artifacts
            if not line or line.startswith("Place") or line.startswith("Minimum") or line.startswith("Maximum") or not line[0].isdigit():
                continue
            
            parts = line.split()
            if not parts:
                continue

            # 1. Parse Place and Car Num from the front (safeguard for ties)
            if len(parts) >= 3 and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            else:
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]

            # 2. Vacuum up all metrics, statuses, and fuel types from the back
            tail_data = []
            while parts and is_efficiency_data(parts[-1]):
                tail_data.insert(0, parts.pop())

            # 3. Whatever is left over perfectly reconstructs the Team Name
            team_name = " ".join(parts)

            # 4. Pad the tail data so it lines up perfectly in the CSV columns
            tail_data += [''] * (8 - len(tail_data))

            # Write the formatted row to the CSV
            row = [place, car_num, team_name] + tail_data
            writer.writerow(row)

    print(f"Success! Efficiency Event data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_efficiency_results('raw-efficiency.txt', 'fsae_efficiency_results.csv')

Success! Efficiency Event data parsed and saved to fsae_efficiency_results.csv


In [7]:
import csv

def is_score(s):
    """Check if a string is a number."""
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_presentation_2024(input_filepath, output_filepath):
    # Added "Status" to the end to catch the "Finalist" tags
    headers = ["Place", "Car Num", "Team", "Raw Score", "Penalty", "Score", "Status"]

    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
        
        writer = csv.writer(outfile)
        writer.writerow(headers)

        for line in infile:
            line = line.strip()
            
            # Skip empty lines or stray header fragments
            if not line or line.startswith("Place") or line.startswith("Printed") or not line[0].isdigit():
                continue
            
            parts = line.split()
            if not parts:
                continue
            
            # 1. Parse Place and Car Num from the front (handles "12 T 23" ties)
            if len(parts) >= 3 and parts[1] == 'T':
                place = f"{parts[0]} T"
                car_num = parts[2]
                parts = parts[3:]
            else:
                place = parts[0]
                car_num = parts[1]
                parts = parts[2:]

            # 2. Check for a text status at the very end of the line (e.g., "Finalist")
            status = ""
            if parts and not is_score(parts[-1]):
                status = parts.pop()

            # 3. Vacuum up the scores from the back
            tail_nums = []
            while parts and is_score(parts[-1]):
                tail_nums.insert(0, parts.pop())

            # 4. Whatever is left over perfectly reconstructs the Team Name
            team_name = " ".join(parts)

            # 5. Route the scores into the correct columns
            raw_score = ""
            penalty = ""
            score = ""

            if len(tail_nums) == 1:
                score = tail_nums[0]
            elif len(tail_nums) == 2:
                # Two numbers means Penalty is blank
                raw_score = tail_nums[0]
                score = tail_nums[1]
            elif len(tail_nums) >= 3:
                # Three numbers means all score columns are filled
                raw_score = tail_nums[0]
                penalty = tail_nums[1]
                score = tail_nums[2]

            # Write the beautifully formatted row
            row = [place, car_num, team_name, raw_score, penalty, score, status]
            writer.writerow(row)

    print(f"Success! 2024 Presentation data parsed and saved to {output_filepath}")

# --- How to use ---
if __name__ == "__main__":
    parse_presentation_2024('2024-raw-presentation.txt', 'fsae_2024_presentation_results.csv')

Success! 2024 Presentation data parsed and saved to fsae_2024_presentation_results.csv
